# ITCS 6169/8169 — Assignment 1: CNN Scene Recognition
**16-class scene recognition · 2,400 training images · transfer learning experiment series**

This notebook documents a systematic experiment series starting from the provided TNet baseline
and iteratively improving through transfer learning, partial fine-tuning, and architecture
selection. Each experiment changes one variable, and decisions are driven by loss-curve
diagnostics. Full rationale and failure analysis are in `EXPERIMENTS.md`.

**Final model:** ConvNeXt-Tiny partial fine-tune · **96.04% val · 96.50% test**

## 1. Setup

In [ ]:
import copy, time, random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from torchvision.models import (
    resnet18, ResNet18_Weights,
    convnext_tiny, ConvNeXt_Tiny_Weights,
)

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
SEED       = 0
BATCH_SIZE = 64
VAL_FRAC   = 0.20

# Update to your assignment directory (Colab: Path('/content/gdrive/MyDrive/assignment1'))
PROJECT_ROOT = Path(r'C:\Users\abhin\Code\ComputerVision')
TRAIN_ROOT   = PROJECT_ROOT / 'data' / 'train'
TEST_ROOT    = PROJECT_ROOT / 'data' / 'test'

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Exp 0 only — grayscale 64×64 matches TNet's expected input
baseline_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

# Standard ImageNet 224×224 — val and test for all Exp 1–6
clean_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Augmented training transform — Exp 2, 4, 5, 6
aug_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [ ]:
# Compute train/val split once — same indices reused across all experiments
_probe     = datasets.ImageFolder(TRAIN_ROOT)
class_names = _probe.classes
num_classes = len(class_names)
n_total     = len(_probe)
val_size    = int(round(n_total * VAL_FRAC))
train_size  = n_total - val_size
del _probe

generator = torch.Generator().manual_seed(SEED)
_split    = random_split(range(n_total), [train_size, val_size], generator=generator)
TRAIN_IDX, VAL_IDX = list(_split[0]), list(_split[1])

print(f"Classes ({num_classes}): {class_names}")
print(f"Train: {train_size}  |  Val: {val_size}")

def make_loaders(train_tf, val_tf):
    """Build train and val loaders using the fixed split indices."""
    tl = DataLoader(Subset(datasets.ImageFolder(TRAIN_ROOT, transform=train_tf), TRAIN_IDX),
                    batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
    vl = DataLoader(Subset(datasets.ImageFolder(TRAIN_ROOT, transform=val_tf),   VAL_IDX),
                    batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    return tl, vl

def make_test_loader(tf):
    return DataLoader(datasets.ImageFolder(TEST_ROOT, transform=tf),
                      batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

# Loaders used across experiments (created once)
baseline_train_loader, baseline_val_loader = make_loaders(baseline_tf, baseline_tf)
clean_train_loader,    clean_val_loader    = make_loaders(clean_tf,    clean_tf)
aug_train_loader,      clean_val_loader2   = make_loaders(aug_tf,      clean_tf)
test_loader = make_test_loader(clean_tf)

print(f"Test images: {len(test_loader.dataset)}")

In [ ]:
@torch.inference_mode()
def evaluate(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, n = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        logits = model(images)
        total_loss += criterion(logits, labels).item() * images.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        n          += labels.size(0)
    return total_loss / n, correct / n


def train_model(model, train_loader, val_loader, optimizer, epochs=20,
                device=device, scheduler=None):
    criterion = nn.CrossEntropyLoss()
    model     = model.to(device)
    best_state, best_val_acc = copy.deepcopy(model.state_dict()), 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    t0 = time.time()

    for epoch in tqdm(range(1, epochs + 1), desc='Epochs', unit='epoch'):
        model.train()
        total_loss, n = 0.0, 0
        for images, labels in tqdm(train_loader, desc=f'  Ep {epoch:02d}', leave=False):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(images), labels)
            loss.backward(); optimizer.step()
            total_loss += loss.item() * images.size(0); n += labels.size(0)

        train_loss = total_loss / n
        val_loss, val_acc = evaluate(model, val_loader, device)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = copy.deepcopy(model.state_dict())
        if scheduler: scheduler.step()

    model.load_state_dict(best_state)
    print(f'Done in {time.time()-t0:.0f}s | Best val acc: {best_val_acc:.4f}')
    return model, history


def plot_history(history, label='', save_path=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(label, fontsize=13)
    ax1.plot(history['train_loss'], label='Train loss')
    ax1.plot(history['val_loss'],   label='Val loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()
    ax2.plot(history['val_acc'], color='green', label='Val accuracy')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_ylim(0, 1); ax2.legend()
    plt.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Best val accuracy: {max(history["val_acc"]):.4f}')


@torch.inference_mode()
def per_class_accuracy(model, loader, class_names, device):
    model.eval()
    correct = torch.zeros(len(class_names))
    total   = torch.zeros(len(class_names))
    for images, labels in loader:
        preds = model(images.to(device, non_blocking=True)).argmax(1).cpu()
        for p, l in zip(preds, labels):
            total[l] += 1
            if p == l: correct[l] += 1
    accs = (correct / total.clamp(min=1)).numpy()
    print(f"\nPer-class accuracy (val set):")
    print(f"{'Class':<22} {'Acc':>6}  Bar"); print('-' * 50)
    for name, acc in sorted(zip(class_names, accs), key=lambda x: x[1]):
        print(f"{name:<22} {acc:>5.1%}  {'█' * int(acc * 20)}")
    print(f"\nMean: {accs.mean():.1%}  Min: {accs.min():.1%}  Max: {accs.max():.1%}")
    return accs

In [ ]:
class TNet(nn.Module):
    """Starter baseline — deliberately minimal (1 conv layer, grayscale 64×64 input)."""
    def __init__(self, num_classes=16):
        super().__init__()
        self.features   = nn.Sequential(nn.Conv2d(1, 16, 3), nn.ReLU(inplace=True), nn.MaxPool2d(4, 4))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(16 * 15 * 15, num_classes))
    def forward(self, x): return self.classifier(self.features(x))

---
## Experiment 0: Baseline TNet

**Hypothesis:** Confirm the pipeline works and establish a floor to beat.

**Config:** TNet (1 conv, 16 channels, MaxPool 4×4) · grayscale 64×64 · Adam lr=0.002 · 20 epochs

**Result:** Best val 49.79% · Final: train loss 0.12, val loss 2.28

**Insight:** Severe overfitting despite a weak model — Adam at lr=0.002 memorizes 1,920 images
without any regularization. Model ceiling is too low to fix; pretrained backbone is the right move.

In [ ]:
set_seed(SEED)
tnet = TNet(num_classes=num_classes)
opt0 = optim.Adam(tnet.parameters(), lr=0.002)
tnet, hist0 = train_model(tnet, baseline_train_loader, baseline_val_loader, opt0, epochs=20, device=device)
plot_history(hist0, label='Exp 0: Baseline TNet', save_path='plots/exp0_baseline.png')

---
## Experiment 1: ResNet18 Frozen Backbone

**Hypothesis:** Pretrained ImageNet features redirected to 16 classes via a new linear head
should substantially outperform TNet, establishing the value of transfer learning alone.

**Change:** ResNet18 pretrained on ImageNet-1K, all conv layers frozen. Only `Linear(512, 16)`
head trained (~8k trainable params). RGB 224×224 with ImageNet normalization.

**Result:** Best val **90.21%** (+40 pts over baseline) · train 0.18, val 0.33

**Insight:** Massive transfer learning gain. Healthy fit — no severe overfitting with only 8k
trainable params. Bottleneck is now representational: a linear head doing linear separation of
ImageNet features not optimized for scene recognition.

In [ ]:
set_seed(SEED)
resnet = resnet18(weights=ResNet18_Weights.DEFAULT)
for p in resnet.parameters(): p.requires_grad = False
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)

print(f"Trainable: {sum(p.numel() for p in resnet.parameters() if p.requires_grad):,} / "
      f"{sum(p.numel() for p in resnet.parameters()):,} params")

opt1 = optim.Adam(resnet.fc.parameters(), lr=1e-3)
resnet, hist1 = train_model(resnet, clean_train_loader, clean_val_loader, opt1, epochs=20, device=device)
plot_history(hist1, label='Exp 1: ResNet18 Frozen', save_path='plots/exp1_resnet18_frozen.png')
per_class_accuracy(resnet, clean_val_loader, class_names, device)

---
## Experiment 2: ResNet18 Frozen + Data Augmentation

**Hypothesis:** Augmenting training exposes the head to more varied scene views, improving
generalization without touching the frozen backbone.

**Change from Exp 1:** Train transform adds RandomResizedCrop, RandomHorizontalFlip,
ColorJitter. Val/test transforms unchanged.

**Result:** Best val **89.17%** (−1.04 vs Exp 1) · slight regression

**Insight:** Augmentation *hurts* on a frozen backbone. The backbone can't adapt to the noisier,
augmented feature vectors — it produces inconsistent 512-dim embeddings from varied crops with no
way to compensate. This is a clean failure case that motivates fine-tuning.

In [ ]:
set_seed(SEED)
resnet2 = resnet18(weights=ResNet18_Weights.DEFAULT)
for p in resnet2.parameters(): p.requires_grad = False
resnet2.fc = nn.Linear(resnet2.fc.in_features, num_classes)

opt2 = optim.Adam(resnet2.fc.parameters(), lr=1e-3)
resnet2, hist2 = train_model(resnet2, aug_train_loader, clean_val_loader, opt2, epochs=20, device=device)
plot_history(hist2, label='Exp 2: ResNet18 Frozen + Augmentation', save_path='plots/exp2_resnet18_aug.png')
per_class_accuracy(resnet2, clean_val_loader, class_names, device)

---
## Experiment 3: ResNet18 Frozen + Cosine LR Scheduling

**Hypothesis:** The flat LR in Exp 1 caused late-epoch oscillation (best checkpoint at ~epoch 13,
regression after). Cosine decay should smooth convergence.

**Change from Exp 1:** Added CosineAnnealingLR(T_max=20). No augmentation — scheduler effect isolated.

**Result:** Best val **89.58%** (−0.63 vs Exp 1) · slight regression

**Insight:** Scheduling hurts on a frozen backbone. The linear head's optimization problem is
simple and fast-converging — cosine decay reduces LR before full convergence. Scheduling pays
off when the loss landscape is complex (i.e., with fine-tuning).

In [ ]:
set_seed(SEED)
resnet3 = resnet18(weights=ResNet18_Weights.DEFAULT)
for p in resnet3.parameters(): p.requires_grad = False
resnet3.fc = nn.Linear(resnet3.fc.in_features, num_classes)

opt3   = optim.Adam(resnet3.fc.parameters(), lr=1e-3)
sched3 = optim.lr_scheduler.CosineAnnealingLR(opt3, T_max=20)
resnet3, hist3 = train_model(resnet3, clean_train_loader, clean_val_loader, opt3, epochs=20,
                              device=device, scheduler=sched3)
plot_history(hist3, label='Exp 3: ResNet18 Frozen + Cosine LR', save_path='plots/exp3_resnet18_cosine.png')
per_class_accuracy(resnet3, clean_val_loader, class_names, device)

---
## Experiment 4: ResNet18 Partial Fine-Tune (layer4 + head)

**Hypothesis:** Unfreezing layer4 lets the highest-level features adapt toward scene
discrimination. With an unlocked backbone, augmentation and cosine scheduling should now help.

**Changes from Exp 1:** layer4 + head trainable (~2.6M params). lr=1e-4 (small to prevent
catastrophic forgetting). Augmented training data. CosineAnnealingLR. 30 epochs.

**Result:** Best val **94.37%** (+4.16 vs Exp 1) · Test **93.00%**

**Insight:** Partial fine-tuning breaks the frozen backbone ceiling. Layer4 adapted quickly —
pretrained weights needed only small adjustments for scenes. Augmentation and scheduling
helped here because the backbone could respond to varied inputs.

In [ ]:
set_seed(SEED)
resnet4 = resnet18(weights=ResNet18_Weights.DEFAULT)
for p in resnet4.parameters(): p.requires_grad = False
for p in resnet4.layer4.parameters(): p.requires_grad = True
resnet4.fc = nn.Linear(resnet4.fc.in_features, num_classes)  # trainable by default

trainable = sum(p.numel() for p in resnet4.parameters() if p.requires_grad)
total     = sum(p.numel() for p in resnet4.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total*100:.1f}%) — layer4 + head")

opt4   = optim.Adam(filter(lambda p: p.requires_grad, resnet4.parameters()), lr=1e-4)
sched4 = optim.lr_scheduler.CosineAnnealingLR(opt4, T_max=30)
resnet4, hist4 = train_model(resnet4, aug_train_loader, clean_val_loader, opt4, epochs=30,
                              device=device, scheduler=sched4)
plot_history(hist4, label='Exp 4: ResNet18 Partial Fine-Tune (layer4)', save_path='plots/exp4_resnet18_partial_finetune.png')
per_class_accuracy(resnet4, clean_val_loader, class_names, device)

---
## Experiment 5: ConvNeXt-Tiny Partial Fine-Tune (best model)

**Hypothesis:** ConvNeXt-Tiny's stronger features (~82% ImageNet top-1 vs ResNet18's ~70%)
should raise the ceiling with the identical partial fine-tuning strategy.

**Changes from Exp 4:** Backbone swapped to ConvNeXt-Tiny. Last two stages (features[6],
features[7]) + classifier head trainable. 768-dim head output. All else identical.

**Why ConvNeXt:** 7×7 depthwise convolutions (larger receptive field), LayerNorm, GELU,
modern ImageNet training recipe → richer, more scene-aware pretrained features.

**Result:** Best val **96.04%** (+1.67 vs Exp 4) · Test **96.50%** ← final model

**Insight:** Plateaued at epoch ~2.5 — pretrained features were already highly scene-relevant,
needed minimal adaptation. Tighter val/train gap (0.134 vs 0.214) confirms better generalization.
Backbone quality is the dominant variable throughout the series.

In [ ]:
set_seed(SEED)
convnext = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
for p in convnext.parameters(): p.requires_grad = False
for p in convnext.features[6].parameters(): p.requires_grad = True
for p in convnext.features[7].parameters(): p.requires_grad = True

in_features = convnext.classifier[2].in_features  # 768
convnext.classifier[2] = nn.Linear(in_features, num_classes)

trainable = sum(p.numel() for p in convnext.parameters() if p.requires_grad)
total     = sum(p.numel() for p in convnext.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total*100:.1f}%) — features[6,7] + head")
print(f"Head input: {in_features}-dim (vs ResNet18's 512-dim)")

opt5   = optim.Adam(filter(lambda p: p.requires_grad, convnext.parameters()), lr=1e-4)
sched5 = optim.lr_scheduler.CosineAnnealingLR(opt5, T_max=30)
convnext, hist5 = train_model(convnext, aug_train_loader, clean_val_loader, opt5, epochs=30,
                               device=device, scheduler=sched5)
plot_history(hist5, label='Exp 5: ConvNeXt-Tiny Partial Fine-Tune', save_path='plots/exp5_convnext_partial_finetune.png')
per_class_accuracy(convnext, clean_val_loader, class_names, device)

---
## Experiment 6: ConvNeXt-Tiny + Deeper Head

**Hypothesis:** A two-layer head with dropout adds non-linear capacity and regularization,
potentially separating hard boundary cases between similar scene classes.

**Change from Exp 5:** Head: `Linear(768, 16)` → `Dropout(0.3) → Linear(768, 256) → GELU → Dropout(0.2) → Linear(256, 16)`.
Backbone and all other settings identical.

**Result:** Best val **95.42%** (−0.62 vs Exp 5) · regression

**Insight:** Head complexity doesn't help when backbone features are already linearly separable.
The added dropout over-regularized the 256-dim layer without geometric benefit. Higher val loss
(0.173 vs 0.134) with similar train loss confirms mild under-fitting from dropout. Exp 5 wins.

In [ ]:
set_seed(SEED)
convnext6 = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
for p in convnext6.parameters(): p.requires_grad = False
for p in convnext6.features[6].parameters(): p.requires_grad = True
for p in convnext6.features[7].parameters(): p.requires_grad = True

in_features = convnext6.classifier[2].in_features
convnext6.classifier[2] = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features, 256),
    nn.GELU(),
    nn.Dropout(p=0.2),
    nn.Linear(256, num_classes),
)

print(f"Head: Dropout(0.3) → Linear(768→256) → GELU → Dropout(0.2) → Linear(256→{num_classes})")

opt6   = optim.Adam(filter(lambda p: p.requires_grad, convnext6.parameters()), lr=1e-4)
sched6 = optim.lr_scheduler.CosineAnnealingLR(opt6, T_max=30)
convnext6, hist6 = train_model(convnext6, aug_train_loader, clean_val_loader, opt6, epochs=30,
                                device=device, scheduler=sched6)
plot_history(hist6, label='Exp 6: ConvNeXt-Tiny + Deeper Head', save_path='plots/exp6_convnext_deeper_head.png')
per_class_accuracy(convnext6, clean_val_loader, class_names, device)

---
## Final Test Evaluation

Model selection was done entirely on the validation set. The test set is evaluated once, here,
after the final model was chosen.

Selected model: **Exp 5 — ConvNeXt-Tiny Partial Fine-Tune** (best val accuracy: 96.04%)

In [ ]:
print('=' * 55)
print('FINAL TEST EVALUATION')
print('=' * 55)

r4_loss, r4_acc = evaluate(resnet4,   test_loader, device)
c5_loss, c5_acc = evaluate(convnext,  test_loader, device)

print(f'\nExp 4 — ResNet18 Partial Fine-Tune    (val 94.37%)')
print(f'  Test accuracy: {r4_acc:.4f}  ({r4_acc*100:.2f}%)')
print(f'  Test loss:     {r4_loss:.4f}')

print(f'\nExp 5 — ConvNeXt-Tiny Partial Fine-Tune (val 96.04%)  ← selected')
print(f'  Test accuracy: {c5_acc:.4f}  ({c5_acc*100:.2f}%)')
print(f'  Test loss:     {c5_loss:.4f}')

print(f'\n{"=" * 55}')
print(f'Final test accuracy: {c5_acc*100:.2f}%')

---
## Ablation Summary

The table below shows the cumulative path to the final model — each row adds one component
to the previous configuration.

| Model Configuration | Val Acc. | Observation |
|---|---:|---|
| Baseline CNN (TNet, from scratch) | 49.79% | Pipeline floor; severe overfitting with 1-layer CNN |
| + Pretrained backbone (ResNet18 frozen) | 90.21% | +40.4 pts — ImageNet features transfer directly |
| + Partial fine-tune + training strategy | 94.37% | +4.2 pts — layer4 unlocked, aug, cosine LR, lr=1e-4 |
| + ConvNeXt-Tiny backbone (final model) | **96.04%** | +1.7 pts — stronger pretrained features |
| **Final model — test set** | **96.50%** | Val and test accuracy consistent; no test-set leakage |

**Failed ablations (components that did not help):**

| Component tested | Val Acc. | vs. baseline config | Reason |
|---|---:|---|---|
| Augmentation on frozen backbone | 89.17% | −1.0 vs Exp 1 | Frozen backbone cannot absorb augmented feature noise |
| Cosine LR on frozen backbone | 89.58% | −0.6 vs Exp 1 | LR decay premature for a convex 8k-param problem |
| Deeper head on ConvNeXt (Exp 6) | 95.42% | −0.6 vs Exp 5 | ConvNeXt features already linearly separable; dropout over-regularizes |

**Key finding:** Backbone quality and partial fine-tuning are the dominant levers.
Augmentation and LR scheduling only help once the backbone is unlocked.